In [1]:
from qdrant_client import QdrantClient
from langchain_groq import ChatGroq

from vectorstore.client_factory import QdrantClientFactory
from vectorstore.qdrant_client import QdrantVectorStore

from retrieval.chapter_fetcher import ChapterContextFetcher
from retrieval.retrieval_service import RetrievalService

from mcq.generator import MCQGenerator
from mcq.validator import MCQValidator
from mcq.consistency_checker import MCQConsistencyChecker
from mcq.faithfulness_checker import MCQFaithfulnessChecker
from mcq.service import MCQService

from mcq.schemas import MCQGenerationRequest

d:\MCQ_BIO\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from config.settings import settings

In [3]:


client = QdrantClientFactory.create()

vector_store = QdrantVectorStore(
    client=client
)

# -------------------------------------------------
# Retrieval Layer
# -------------------------------------------------

chapter_fetcher = ChapterContextFetcher(
    vector_store=vector_store
)

retrieval_service = RetrievalService(
    chapter_fetcher=chapter_fetcher
)

# -------------------------------------------------
# LLM
# -------------------------------------------------

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.7,
    api_key= settings.GROQ_API_KEY
)

# -------------------------------------------------
# MCQ Components
# -------------------------------------------------

generator = MCQGenerator(
    llm=llm
)

validator = MCQValidator()

consistency_checker = MCQConsistencyChecker(
    llm=llm
)

faithfulness_checker = MCQFaithfulnessChecker(
    llm=llm
)

# -------------------------------------------------
# MCQ Service
# -------------------------------------------------

mcq_service = MCQService(
    retrieval_service=retrieval_service,
    generator=generator,
    validator=validator,
    consistency_checker=consistency_checker,
    faithfulness_checker=faithfulness_checker
)

In [12]:
request = MCQGenerationRequest(
    class_number=11,
    chapter="Photosynthesis in Higher Plants",
    page=1,
    num_questions=30
)

In [13]:
result = mcq_service.generate_mcqs(request)

In [14]:
print(f"Generated MCQs: {len(result.mcqs)}")

Generated MCQs: 31


In [15]:
for i, mcq in enumerate(result.mcqs, start=1):
    print("=" * 80)

    print(f"Q{i}. {mcq.question}")
    print()

    print(f"1. {mcq.option_1}")
    print(f"2. {mcq.option_2}")
    print(f"3. {mcq.option_3}")
    print(f"4. {mcq.option_4}")

    print()
    print(f"Correct Option: {mcq.correct_option}")
    print()
    print(f"Explanation: {mcq.explanation}")
    print()

Q1. What is the primary source of energy for all living forms on earth?

1. Respiration
2. Photosynthesis
3. Decomposition
4. Fermentation

Correct Option: 2

Explanation: The primary source of energy for all living forms on earth is photosynthesis, as it is the process by which green plants use light energy to drive the synthesis of organic compounds.

Q2. Who proposed that plants change light energy to chemical energy by transferring an electron in an organised array of pigment molecules and other substances?

1. Joseph Priestley
2. Melvin Calvin
3. Jan Ingenhousz
4. Julius von Sachs

Correct Option: 2

Explanation: Melvin Calvin proposed that plants change light energy to chemical energy by transferring an electron in an organised array of pigment molecules and other substances.

Q3. What is the term for the ability of plants to make their own food through photosynthesis?

1. Heterotrophy
2. Autotrophy
3. Phototrophy
4. Chemotrophy

Correct Option: 2

Explanation: Autotrophy refers 